# Ingestion 1: replayable batch extraction

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/ingestion/ingestion-01-batch-patterns.ipynb)

Use a stable cursor and keyed writes to make a batch safe to replay.

## Setup and sample data

Two events share a timestamp. An ID breaks the tie in the checkpoint.

In [ ]:
source = [("09:00", 1), ("09:00", 2), ("09:01", 3)]
checkpoint = ("09:00", 1)

## Page over a stable order

A batch reader can advance a cursor made of a timestamp and a stable key. The source must return rows in the same order.

**Where it matters**

Read database changes with a composite high-water mark.

**Watch out for**

Late rows can fall below the cursor. Keep a lookback with keyed writes or use a durable change log.

In [ ]:
page = [row for row in sorted(source) if row > checkpoint]
print("page:", page)
print("next cursor:", page[-1] if page else checkpoint)

**What to notice:** ID 2 is included even though its timestamp equals the checkpoint timestamp.

## Advance after the sink commits

Apply a batch with a deterministic key and save progress after the write succeeds.

**Where it matters**

Make reruns safe when the worker fails after publishing.

**Watch out for**

A mutable source must supply a stable version to prevent stale updates from replacing newer values.

In [ ]:
sink = {}
for attempt in range(2):
    sink.update({order_id: timestamp for timestamp, order_id in page})
    print("attempt", attempt + 1, "rows:", sorted(sink))

**What to notice:** Both writes leave the same two keys in the sink.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Resume from (09:00, 2). Expect only ID 3.

In [ ]:
# Try your solution here.

In [ ]:
print([row for row in source if row > ("09:00", 2)])

## Key takeaways

- Read database changes with a composite high-water mark.
- Make reruns safe when the worker fails after publishing.

**Next:** [Ingestion 2: APIs](ingestion-02-apis.ipynb)